In [ ]:
### Run this cell to download and install the necessary modules for the homework
!pip install datasets tiktoken huggingface_hub
!pip install --upgrade --no-deps git+https://github.com/locuslab/mugrade.git
# !wget -nc https://raw.githubusercontent.com/modernaicourse/hw7/refs/heads/main/hw7_tests.py

from datasets import load_dataset
import os
import math
import mugrade
import torch
import tiktoken
from torch.nn import Module, ModuleList, Parameter, Buffer
import json
import copy
import re
import warnings

from hw7_tests import (test_Linear,
                       test_Embedding,
                       test_silu,
                       test_rms_norm,
                       test_self_attention,
                       test_MLP,
                       test_TransformerBlock,
                       test_Adam,
                       test_LLM,
                       test_log_probs,
                       test_MultiHeadAttentionKVCache, submit_MultiHeadAttentionKVCache,
                       test_generate_parallel, submit_generate_parallel,
                       test_gsm8k_to_text, submit_gsm8k_to_text,
                       test_pretokenize_gsm8k, submit_pretokenize_gsm8k,
                       test_get_loss_mask, submit_get_loss_mask,
                       test_DataLoader, submit_DataLoader,
                       test_eval_tool, submit_eval_tool,
                       test_generate_parallel_tool, submit_generate_parallel_tool,
                       test_extract_answer, submit_extract_answer,
                       test_grade_responses, submit_grade_responses,
                       test_eval_model, submit_eval_model,
                       test_rl_loss, submit_rl_loss,
                       test_train_llm_rl, submit_train_llm_rl)

In [ ]:
# Select the accelerator available in this runtime.
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## Homework 7 - Reasoning models and Reinforcement Learning

In this homework, you will use reinforcement learning, starting from a supervised-finetuned checkpoint we provide, to build a (very minimal, of course) reasoning model that can (sometimes) solve basic math problems and employ simple tool use.  While the actual performance of your model is going to pale in comparison to what even slightly larger LLMs can do, it will be a nice illustration of what is possible using _just_ the code that you write in this notebook, with the caveat that the pretrained model you start from was trained for far longer than would be practical within a single assignment.

As a preview of what you will implement, you will finetune an LLM that can solve problems like this one, from the GSM8K dataset.
```
Toulouse has twice as many sheep as Charleston. Charleston has 4 times as many sheep as Seattle. How many sheep do Toulouse, Charleston, and Seattle have together if Seattle has 20 sheep?
```

The final performance of your model may vary quite a bit (RL training in particular can be noisy), but you should expect to be able to build a system that can solve this kind of problem >3% of the time, and can solve it >30% of the time if given 32 attempts.  What's more, to do so, the generations will explicitly call out to Python, with proper arithmetic formatting, for simple arithmetic operations.

As with the model code throughout this course, you should use **no** routines from the `torch.nn` library except for the `Module`, `ModuleList`, `Parameter`, and `Buffer` classes imported in the cell above.

## Part 0 - The model and training components

This assignment builds a reasoning model on top of an ordinary decoder-only transformer, so we begin by writing every piece of that model and of the optimization machinery that trains it.  None of the components in this part are specific to reasoning or RL: they are the basic layers of the network (`Linear`, `Embedding`, `silu`, `rms_norm`, `self_attention`, and `MLP`), the two loss functions we need (`cross_entropy_loss` and `log_probs`), and the `Adam` optimizer.

Each of these cells has a _local_ test only, so you can check your implementation as you go, but there is nothing to submit to mugrade for them; the graded questions all start in Part 1.

The one remaining piece of the network, the multi-head attention layer, needs a small modification for this assignment, so we implement it -- and then assemble the block and the full model out of all these pieces -- at the start of Part 1.

### Linear layer

Implement a linear layer with no bias term, which computes
$$ \mathrm{Linear}(X) = X W^T $$
for a weight matrix $W \in \mathbb{R}^{d_{\mathrm{out}} \times d_{\mathrm{in}}}$.  Because we are going to train (and further finetune) these networks, the weights do need to be properly initialized: use random normal entries scaled by $\sqrt{2/d_{\mathrm{in}}}$.

Store the weights in a `Parameter` named `weight`, of size `(out_dim, in_dim)`, as both the tests and the checkpoint-loading code below rely on this name and ordering.

In [ ]:
@mugrade.local_tests
class Linear(Module):
    """ Linear layer with no bias term.  The parameters of the layer are stored in a .weight Parameter"""
    def __init__(self, in_dim, out_dim):
        """
        Initialize a linear layer without a bias term.

        Inputs:
            in_dim : int - input feature dimension
            out_dim : int - output feature dimension
        """
        super().__init__()
        ### BEGIN YOUR CODE
        self.weight = Parameter(torch.randn(out_dim, in_dim) * math.sqrt(2 / in_dim))
        ### END YOUR CODE

    def forward(self, X):
        """
        Apply the linear layer to one or more input vectors.

        Input:
            X : torch.Tensor[float] (... x in_dim) - input tensor
        Output:
            torch.Tensor[float] (... x out_dim) - transformed tensor
        """
        ### BEGIN YOUR CODE
        return torch.matmul(X, self.weight.t())
        ### END YOUR CODE

### Embedding layer

Next, implement an embedding layer.  This is the layer that converts the one-hot encoding of the input into a matrix of input embeddings, e.g., by effectively computing the linear operation $X_{\text{one-hot}} W_E$ where $W_E \in \mathbb{R}^{V \times d}$ is a matrix of embeddings for each of the $V$ tokens (for a few minor technical reasons we won't get in to, it's common to represent $W_E$ without the transpose for embedding layers, e.g., this is what PyTorch does).

However, note that the input to this layer is not an actual one-hot embedding matrix (it would be very inefficient to store this very sparse matrix explicitly).  Instead, the input is an integer tensor $Y$ that stores the _index_ of each non-zero entry in the one-hot embedding.  That is, if the first row of $X_{\text{one-hot}}$ would contain a one in position $i$ and zeros everywhere else, then the first element of $Y$ would be just the integer $i$ (i.e., the elements of $Y$ can range from 0 to `num_tokens-1`).  $Y$ can have any size, though in the common usage of the function, the dimensions would be `(batch_size, seq_len)` and the layer would return a tensor of size `(batch_size, seq_len, dim)`.

Represent the weights with a `Parameter` named `weight` of size `(num_tokens, dim)`, initialized to (unscaled) random normal entries.

In [ ]:
@mugrade.local_tests
class Embedding(Module):
    def __init__(self, num_tokens, dim):
        """
        Initialize an embedding table over a fixed vocabulary.

        Inputs:
            num_tokens : int - vocabulary size
            dim : int - embedding dimension
        """
        super().__init__()
        ### BEGIN YOUR CODE
        self.weight = Parameter(torch.randn(num_tokens, dim))
        ### END YOUR CODE

    def forward(self, Y):
        """
        Look up embeddings for an integer tensor of token ids.

        Input:
            Y : torch.Tensor[int] (...) - tensor of token indices in [0, num_tokens)
        Output:
            torch.Tensor[float] (... x dim) - embedding vectors for each token id
        """
        ### BEGIN YOUR CODE
        return self.weight[Y]
        ### END YOUR CODE

### SiLU nonlinearity

Implement the SiLU nonlinearity, applied elementwise, defined as
$$\mathrm{silu}(x) = x \cdot \mathrm{sigmoid}(x).$$

In [ ]:
@mugrade.local_tests
def silu(x):
    """
    Apply the SiLU nonlinearity elementwise.

    Input:
        x : torch.Tensor[float] (...) - input tensor
    Output:
        torch.Tensor[float] (...) - tensor after applying SiLU
    """
    ### BEGIN YOUR CODE
    return x * torch.sigmoid(x)
    ### END YOUR CODE

### RMS norm

Implement a normalization layer, usually referred to as "RMSNorm", applied along the _last_ dimension of the input:
$$ \mathrm{rms\_norm}(x) = \frac{x}{\sqrt{\frac{1}{d}\|x\|_2^2 + \epsilon}} $$
where $d$ is the size of that last dimension, and $\epsilon$ is a small constant for numerical stability.

Note that many presentations of RMSNorm (and the actual Llama models) include a learned scaling weight that multiplies this quantity elementwise.  We leave it out here, since the per-norm scaling doesn't really make any difference for the size of network we are considering, which means we can implement the normalization as a plain function rather than as a Module with parameters.

In [ ]:
@mugrade.local_tests
def rms_norm(X, eps=1e-5):
    """
    Apply RMS normalization along the final dimension of the input.

    Inputs:
        X : torch.Tensor[float] (... x dim) - input tensor
        eps : float - numerical stability constant
    Output:
        torch.Tensor[float] (... x dim) - RMS-normalized tensor
    """
    ### BEGIN YOUR CODE
    norm = torch.sqrt(torch.mean(X ** 2, dim=-1, keepdim=True) + eps)
    return X / norm
    ### END YOUR CODE

### Masked self attention

Implement a (masked) self attention operation:
$$Y = \mathrm{softmax} \left ( \frac{ Q K^T}{\sqrt{d}} + \mathrm{M} \right ) V$$
where (in mathematical notation) $Q \in \mathbb{R}^{T_q \times d}$, $K,V \in \mathbb{R}^{T_k \times d}$, and $M \in \mathbb{R}^{T_q \times T_k}$, and where the softmax is applied to the last dimension.  This should work both for the case above where $Q,K,V$ are all matrices (2D tensors), _and_ for the case where there are additional leading dimensions in the tensors, such as a batch dimension or an additional head dimension for multi-head attention.  If `mask` is `None`, no mask is added at all.

In [ ]:
@mugrade.local_tests
def self_attention(Q,K,V, mask=None):
    """
    Apply scaled dot-product attention, optionally with an additive mask.

    Inputs:
        Q : torch.Tensor[float] (... x query_len x d) - query tensor
        K : torch.Tensor[float] (... x key_len x d) - key tensor
        V : torch.Tensor[float] (... x key_len x d_v) - value tensor
        mask : torch.Tensor[float] (... x query_len x key_len) or None - additive attention mask
    Output:
        torch.Tensor[float] (... x query_len x d_v) - attention output tensor
    """
    ### BEGIN YOUR CODE
    d = Q.shape[-1]
    attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d)
    if mask is not None:
        attn_scores = attn_scores + mask
    attn_weights = torch.softmax(attn_scores, dim=-1)
    return torch.matmul(attn_weights, V)
    ### END YOUR CODE

### Feed-forward network

Each transformer block uses a simple two-layer MLP
$$ \mathrm{MLP}(X) = \mathrm{silu}(X W_1^T) W_2^T $$
where $W_1 \in \mathbb{R}^{d_{\mathrm{ffn}} \times d}$ and $W_2 \in \mathbb{R}^{d \times d_{\mathrm{ffn}}}$.  (The Llama class of models instead uses a "gated" MLP with a third weight matrix, but there isn't any real advantage to the gated version for the size of network we're using here.)

Store the two layers as `Linear` modules with the names `w1` and `w2`.

In [ ]:
@mugrade.local_tests
class MLP(Module):
    def __init__(self, dim, ffn_dim):
        """
        Initialize a simple two layer feed-forward network used in the transformer block.

        Inputs:
            dim : int - model dimension
            ffn_dim : int - hidden feed-forward dimension
        """
        super().__init__()
        ### BEGIN YOUR CODE
        self.w1 = Linear(dim, ffn_dim)
        self.w2 = Linear(ffn_dim, dim)
        ### END YOUR CODE

    def forward(self, X):
        """
        Apply a simple two layer feed-forward network to the input tensor.

        Input:
            X : torch.Tensor[float] (... x dim) - input tensor
        Output:
            torch.Tensor[float] (... x dim) - transformed tensor
        """
        ### BEGIN YOUR CODE
        return self.w2(silu(self.w1(X)))
        ### END YOUR CODE

### Masked sequence log probabilities

Implement a function that takes a batch of `logits`, the target next-token ids `y`, and a boolean `mask`, and returns, for each element $i$ of the batch, the _sum_ of the log probabilities assigned to the target tokens over just those positions where the mask is `True`:
$$ \text{log\_probs}_i = \sum_{t=1}^{T} \mathrm{mask}_{i,t} \cdot \log \mathrm{softmax}(\text{logits}_{i,t})_{y_{i,t}}. $$
The output is therefore a vector with one entry per batch element, not a single number for the whole batch.  Use `torch.log_softmax()` rather than taking the log of a softmax, for numerical stability.

In [ ]:
@mugrade.local_tests
def log_probs(logits, y, mask):
    """
    Compute masked sequence log probabilities for each batch element.

    Inputs:
        logits : torch.Tensor[float] (batch_size x seq_len x num_tokens) - predicted logits
        y : torch.Tensor[int] (batch_size x seq_len) - desired next-token ids
        mask : torch.Tensor[bool] (batch_size x seq_len) - mask selecting tokens to include
    Output:
        torch.Tensor[float] (batch_size,) - summed masked log probabilities per example
    """
    ### BEGIN YOUR CODE
    log_p = torch.log_softmax(logits, dim=-1) # (batch_size x seq_len x num_tokens)
    log_p_y = log_p.gather(-1, y.unsqueeze(-1)).squeeze(-1) # (batch_size x seq_len)
    return (log_p_y * mask).sum(dim=-1) # (batch_size)
    ### END YOUR CODE

### Adam

Implement the Adam optimizer.  Recall that the Adam updates are given by:
$$
\begin{split}
u & := \beta_1 u + (1-\beta_1) \nabla_W \text{Loss} \\
v & := \beta_2 v + (1-\beta_2) \nabla_W \text{Loss}^2 \quad \text{(square applied elementwise)} \\
\hat{u} & := u / (1 - \beta_1^t) \\
\hat{v} & := v / (1 - \beta_2^t) \\
w & := w - \eta \frac{\hat{u}}{\sqrt{\hat{v}} + \epsilon}  \quad \text{(division done elementwise)}  \\
\end{split}
$$
where $t$ counts the number of updates taken so far (starting at 1 for the first update), and where $u$ and $v$ are both initialized to zero.

The optimizer is a plain Python class (not a `Module`) with three methods: `__init__()`, `step()` (which updates the parameters in place), and `zero_grad()` (which clears the gradients of all parameters).  Two common pitfalls to keep in mind when implementing an optimizer this way:
- In your `__init__()` function, you should explicitly call `list()` on the `params` input to store it in your class (and form similar lists for the `u` and `v` moment estimates).  This is because the `model.parameters()` function returns a Python generator, an object that can be iterated over _one_ time to return all its elements.  So if you only store the passed `params` variable and then try to iterate over it during your `zero_grad()` or `step()` functions, you will only iterate over the parameters one time, and thereafter there won't be any elements to iterate over.
- You need to compute the updates to the parameters within a `torch.no_grad()` block, as shown below.  The reason for this is that otherwise, the gradient update will happen _within the automatic differentiation loop itself_, i.e., you will be computing the gradient of the entire chain of parameter updates you perform with gradient descent.  There are actually some very cool reasons why it's often useful to differentiate through an entire parameter update, but that is definitely not what we want here.
```python
with torch.no_grad():
    ### parameter update here
```
Finally, note that a parameter may have no gradient at all (i.e., `p.grad` is `None`) if it wasn't used in the loss, so both `step()` and `zero_grad()` should simply skip those parameters.

In [ ]:
@mugrade.local_tests
class Adam:
    def __init__(self, params, lr=1e-3, betas = (0.9, 0.999), eps=1e-8):
        """
        Initialize Adam optimizer state for a set of parameters.

        Inputs:
            params : iterable[torch.nn.Parameter] - parameters to optimize
            lr : float - learning rate
            betas : tuple(float, float) - decay rates for first and second moments
            eps : float - numerical stability constant
        """
        ### BEGIN YOUR CODE
        self.params = list(params)
        self.lr = lr
        self.beta1, self.beta2 = betas
        self.eps = eps
        self.t = 0
        self.u = [torch.zeros_like(p) for p in self.params]
        self.v = [torch.zeros_like(p) for p in self.params]
        ### END YOUR CODE

    def step(self):
        """
        Apply one Adam update to all stored parameters.
        """
        ### BEGIN YOUR CODE
        self.t += 1
        with torch.no_grad():
            for i, p in enumerate(self.params):
                if p.grad is None:
                    continue
                self.u[i] = self.beta1 * self.u[i] + (1 - self.beta1) * p.grad
                self.v[i] = self.beta2 * self.v[i] + (1 - self.beta2) * p.grad ** 2
                u_hat = self.u[i] / (1 - self.beta1 ** self.t)
                v_hat = self.v[i] / (1 - self.beta2 ** self.t)
                p -= self.lr * u_hat / (torch.sqrt(v_hat) + self.eps)
        ### END YOUR CODE

    def zero_grad(self):
        """
        Zero out gradients for all stored parameters when gradients exist.
        """
        ### BEGIN YOUR CODE
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()
        ### END YOUR CODE

## Part 1 - Parallel Sampling

When we work on RL methods, which generally require generating (many) samples from the underlying model in order to improve performance, it is extremely helpful to be able to simultaneously sample _multiple_ different completions from the same prompt.  A KV cache is usually written to store the keys and values of a _single_ sequence, since generation is most often done one sequence at a time; here, we instead need a cache that can hold a whole batch of sequences at once, so that all the completions of a single prompt can be generated together in one set of forward passes.

### Multi-batch KV Cache

Implement the `MultiHeadAttentionKVCache` class, which performs multi-head self attention using a _batched_ key/value cache.  The forward pass should perform the following steps:
1. Form the $Q,K,V$ tensors by applying the `Linear` modules `wq`, `wk`, and `wv` to the input (create these within the class using exactly these names).
2. If `use_kv_cache` is `True`, write `K` and `V` into positions `seq_pos:seq_pos+seq_len` (along the sequence dimension) of the caches, for the first `batch_size` rows of the caches.  Then set the `K` and `V` used for attention to be the entire cache, over these same `batch_size` rows, up to position `seq_pos+seq_len`.  Updating the cache in place like this, rather than literally keeping a fixed cache and concatenating to it, allows you to re-use the same cache across multiple generations without having to reset it.
3. Split $Q,K,V$ along the last dimension to create `n_heads` different heads
   $$
   Q = \left [\begin{array}{cccc} Q_1 & Q_2 & \cdots & Q_h \end{array} \right ], \;\; K = \left [\begin{array}{cccc} K_1 & K_2 & \cdots & K_h \end{array} \right ], \;\; V = \left [\begin{array}{cccc} V_1 & V_2 & \cdots & V_h \end{array} \right ]$$
   where each $Q_i,K_i,V_i$ is a sub-block of trailing dimension `dim / n_heads`.
4. Apply the `self_attention()` function to each head and concatenate the results back together
   $$Y = \left [ \begin{array}{cccc} \text{SelfAttn}(Q_1,K_1,V_1)  & \cdots & \text{SelfAttn}(Q_h,K_h,V_h) \end{array} \right ]$$
   You are welcome to implement this _either_ by explicitly splitting the tensor along the head dimension and using a for loop, _or_ by using tensor reshaping to make this operation more efficient.
5. Apply a final projection, a `Linear` module named `wp`, to $Y$.

The caches themselves should be created in `__init__()` as PyTorch `Buffer` objects named `.k_cache` and `.v_cache` (wrapping them in `Buffer` ensures they are converted to the proper datatype, and moved to the GPU, along with the rest of the model).  The important point for this assignment is their size: each should be of size `(max_cache_batches x max_cache_size x dim)`, so that the layer caches an entire batch of up to `max_cache_batches` sequences rather than just a single one.  The forward call should then store the KV cache for however large the input batch size happens to be.

In [ ]:
@mugrade.local_tests
class MultiHeadAttentionKVCache(Module):
    def __init__(self, dim, n_heads, max_cache_size, max_cache_batches=64):
        """
        Initialize a multi-head self-attention layer with KV cache buffers.

        Inputs:
            dim : int - total embedding dimension
            n_heads : int - number of attention heads
            max_cache_size : int - maximum sequence length stored in the cache
            max_cache_batches : int - maximum batch size for cache
        """
        ### BEGIN YOUR CODE
        super().__init__()
        assert dim % n_heads == 0, "dim must be divisible by n_heads"
        self.dim = dim
        self.n_heads = n_heads
        self.d_head = dim // n_heads
        self.wq = Linear(dim, dim)
        self.wk = Linear(dim, dim)
        self.wv = Linear(dim, dim)
        self.wp = Linear(dim, dim)
        # the caches now hold up to max_cache_batches sequences instead of just one
        self.k_cache = Buffer(torch.zeros(max_cache_batches, max_cache_size, dim))
        self.v_cache = Buffer(torch.zeros(max_cache_batches, max_cache_size, dim))
        ### END YOUR CODE

    def forward(self, X, mask=None, seq_pos=0, use_kv_cache=False):
        """
        Apply multi-head self-attention, optionally updating and using the KV cache.

        Inputs:
            X : torch.Tensor[float] (batch_size x seq_len x dim) - input sequence embeddings
            mask : torch.Tensor[float] (seq_len x total_len) or None - additive attention mask
            seq_pos : int - starting sequence position for cached tokens
            use_kv_cache : bool - whether to update and use the KV cache
        Output:
            torch.Tensor[float] (batch_size x seq_len x dim) - attention output
        """
        ### BEGIN YOUR CODE
        batch_size, seq_len, _ = X.shape
        Q = self.wq(X) # (batch_size x seq_len x dim)
        K = self.wk(X) # (batch_size x seq_len x dim)
        V = self.wv(X) # (batch_size x seq_len x dim)
        if use_kv_cache:
            # cache the new keys/values for just this batch, then attend over everything cached so far
            self.k_cache[:batch_size, seq_pos:seq_pos+seq_len] = K
            self.v_cache[:batch_size, seq_pos:seq_pos+seq_len] = V
            K = self.k_cache[:batch_size, :seq_pos+seq_len]
            V = self.v_cache[:batch_size, :seq_pos+seq_len]
        Q = Q.view(batch_size, -1, self.n_heads, self.d_head).transpose(1, 2) # (batch_size x n_heads x seq_len x d_head)
        K = K.view(batch_size, -1, self.n_heads, self.d_head).transpose(1, 2) # (batch_size x n_heads x total_len x d_head)
        V = V.view(batch_size, -1, self.n_heads, self.d_head).transpose(1, 2) # (batch_size x n_heads x total_len x d_head)
        attn_output = self_attention(Q, K, V, mask) # (batch_size x n_heads x seq_len x d_head)
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.dim) # (batch_size x seq_len x dim)
        return self.wp(attn_output) # (batch_size x seq_len x dim)
        ### END YOUR CODE

### Transformer block

With the attention layer in place, we can assemble the rest of the network.  Implement a transformer block, defined as the operation
$$\begin{split}
Z &= X + \mathrm{MHA}(\mathrm{rms\_norm}(X)) \\
Y &= Z + \mathrm{MLP}(\mathrm{rms\_norm}(Z))
\end{split}$$
where MHA denotes the multi-head attention layer (with KV cache) you just implemented.  Store the attention layer and the MLP in the class with the names `attn` and `mlp`, and pass the `mask`, `seq_pos`, and `use_kv_cache` arguments through to the attention layer.  Since `rms_norm()` here is just a function with no weights, there is nothing else to store in the class.

Note that the block passes its own `max_cache_size` down to the attention layer, but leaves `max_cache_batches` at its default, which is what determines the largest number of completions we can generate in parallel later on.

In [ ]:
@mugrade.local_tests
class TransformerBlock(Module):
    def __init__(self, dim, n_heads, ffn_dim, max_cache_size):
        """
        Initialize a transformer block with multi-head attention and feed-forward layers.

        Inputs:
            dim : int - model dimension
            n_heads : int - number of attention heads
            ffn_dim : int - hidden feed-forward dimension
            max_cache_size : int - maximum sequence length stored in the attention cache
        """
        super().__init__()
        ### BEGIN YOUR CODE
        self.attn = MultiHeadAttentionKVCache(dim, n_heads, max_cache_size)
        self.mlp = MLP(dim, ffn_dim)
        ### END YOUR CODE

    def forward(self, X, mask=None, seq_pos=0, use_kv_cache=False):
        """
        Apply one transformer block with residual connections.

        Inputs:
            X : torch.Tensor[float] (batch_size x seq_len x dim) - input sequence embeddings
            mask : torch.Tensor[float] (seq_len x total_len) or None - additive attention mask
            seq_pos : int - starting sequence position for cached tokens
            use_kv_cache : bool - whether to update and use the attention cache
        Output:
            torch.Tensor[float] (batch_size x seq_len x dim) - transformed sequence embeddings
        """
        ### BEGIN YOUR CODE
        X = X + self.attn(rms_norm(X), mask, seq_pos, use_kv_cache)
        X = X + self.mlp(rms_norm(X))
        return X
        ### END YOUR CODE

### The full language model

Finally, put the pieces together into the full model.  The model needs to contain the following elements:
- `.embedding`: an `Embedding` layer that maps integers of max value `num_tokens` to the embedding dimension `dim`
- `.pos_embeddings`: a `(max_seq_len, dim)` PyTorch `Parameter` that stores the (learned) global positional embeddings, initialized to random normal entries
- `.layers`: a PyTorch `ModuleList` of length `num_layers`, where each element is a `TransformerBlock` (with a cache size of `max_seq_len`)
- `.output`: a `Linear` layer that converts `dim` dimensional inputs to `num_tokens` dimensional outputs
- `.mask`: a PyTorch `Buffer` containing a `(max_seq_len, max_seq_len)` strictly upper triangular mask (negative infinity in the strictly upper diagonal entries, zero elsewhere).  You will pass subsets of this mask to the transformer layers based upon the actual sequence position of the token inputs.

The full set of operations performed by the model in the forward pass should be the following:
1. Apply the `.embedding` layer to the input `tokens`, and add `.pos_embeddings` for the appropriate sequence positions (i.e., starting at `seq_pos`).
2. Apply each element of `.layers` sequentially, using the appropriate subset of the mask: rows `seq_pos:seq_pos+seq_len` and columns `:seq_pos+seq_len`, since the tokens we are processing now can attend to everything already in the cache.
3. Return `.output` applied to `rms_norm()` of the output of the last layer.

We also include (you don't need to write these, but it's worth checking that your objects are all named correctly) `load_weights()` and `save_weights()` functions, which read and write checkpoints in a Llama-like format.  These are how we load the pretrained model in the next cell, and how you will save the finetuned models you train later in the assignment.

In [ ]:
@mugrade.local_tests
class LLM(Module):
    def __init__(self, num_tokens, dim, n_heads, max_seq_len, ffn_dim, num_layers):
        """
        Initialize the simplified Llama 3 model used in this homework.

        Inputs:
            num_tokens : int - vocabulary size
            dim : int - model dimension
            n_heads : int - number of attention heads per layer
            max_seq_len : int - maximum supported sequence length
            ffn_dim : int - hidden feed-forward dimension in each block
            num_layers : int - number of transformer blocks
        """
        super().__init__()
        ### BEGIN YOUR CODE
        self.embedding = Embedding(num_tokens, dim)
        self.pos_embeddings = Parameter(torch.randn(max_seq_len, dim))
        self.layers = ModuleList([TransformerBlock(dim, n_heads, ffn_dim, max_seq_len) for _ in range(num_layers)])
        self.output = Linear(dim, num_tokens)
        self.mask = Buffer(torch.triu(torch.full((max_seq_len, max_seq_len), float('-inf')), diagonal=1))
        ### END YOUR CODE

    def forward(self, tokens, seq_pos=0, use_kv_cache=False):
        """
        Apply the full language model to a batch of token sequences.

        Inputs:
            tokens : torch.Tensor[int] (batch_size x seq_len) - input token ids
            seq_pos : int - starting sequence position for positional embeddings and cached attention
            use_kv_cache : bool - whether to update and use cached keys and values
        Output:
            torch.Tensor[float] (batch_size x seq_len x num_tokens) - output logits
        """
        ### BEGIN YOUR CODE
        seq_len = tokens.shape[1]
        X = self.embedding(tokens) # (batch_size x seq_len x dim)
        X = X + self.pos_embeddings[seq_pos:seq_pos+seq_len]
        mask = self.mask[seq_pos:seq_pos+seq_len, :seq_pos+seq_len] # (seq_len x total_len)
        for layer in self.layers:
            X = layer(X, mask, seq_pos=seq_pos, use_kv_cache=use_kv_cache)
        return self.output(rms_norm(X)) # (batch_size x seq_len x num_tokens)
        ### END YOUR CODE

    def load_weights(self, filename):
        """ Load a model from a Llama-like checkpoint """
        checkpoint = torch.load(filename, map_location = self.embedding.weight.device)
        self.embedding.weight.data = checkpoint["tok_embeddings.weight"]
        self.pos_embeddings.data = checkpoint["pos_embeddings.weight"]
        self.output.weight.data = checkpoint["output.weight"]

        for i,layer in enumerate(self.layers):
            layer.attn.wq.weight.data = checkpoint[f"layers.{i}.attention.wq.weight"]
            layer.attn.wk.weight.data = checkpoint[f"layers.{i}.attention.wk.weight"]
            layer.attn.wv.weight.data = checkpoint[f"layers.{i}.attention.wv.weight"]
            layer.attn.wp.weight.data = checkpoint[f"layers.{i}.attention.wo.weight"]

            layer.mlp.w1.weight.data = checkpoint[f"layers.{i}.feed_forward.w1.weight"]
            layer.mlp.w2.weight.data = checkpoint[f"layers.{i}.feed_forward.w2.weight"]

    def save_weights(self, filename):
        """ Save a model to a Llama-like checkpoint."""
        checkpoint = {}
        checkpoint["tok_embeddings.weight"] = self.embedding.weight.detach().to(torch.bfloat16).cpu()
        checkpoint["pos_embeddings.weight"] = self.pos_embeddings.detach().to(torch.bfloat16).cpu()
        checkpoint["output.weight"] = self.output.weight.detach().to(torch.bfloat16).cpu()

        for i, layer in enumerate(self.layers):
            checkpoint[f"layers.{i}.attention.wq.weight"] = layer.attn.wq.weight.detach().to(torch.bfloat16).cpu()
            checkpoint[f"layers.{i}.attention.wk.weight"] = layer.attn.wk.weight.detach().to(torch.bfloat16).cpu()
            checkpoint[f"layers.{i}.attention.wv.weight"] = layer.attn.wv.weight.detach().to(torch.bfloat16).cpu()
            checkpoint[f"layers.{i}.attention.wo.weight"] = layer.attn.wp.weight.detach().to(torch.bfloat16).cpu()
            checkpoint[f"layers.{i}.feed_forward.w1.weight"] = layer.mlp.w1.weight.detach().to(torch.bfloat16).cpu()
            checkpoint[f"layers.{i}.feed_forward.w2.weight"] = layer.mlp.w2.weight.detach().to(torch.bfloat16).cpu()
        torch.save(checkpoint, filename)

### Parallel Generation

Implement the `generate_parallel()` function below, which for a _single_ prompt generates `num_completions` different completions, all at the same time.  The process works as follows:
1. Create a `(num_completions x max_tokens)` integer tensor to hold the tokens, on the same device as the model (which you can get via `next(model.parameters()).device`), and copy the prompt into the first `len(prompt_tokens)` entries of every row.
2. Run the model on the full prompt (with `use_kv_cache=True`) to get the logits for the next token of each completion, and sample the next token of each row from the softmax distribution over these logits, divided by `temp`.
3. Repeatedly feed just the newly generated token back into the model (again using the KV cache, and advancing `seq_pos` by the number of tokens you just processed), sample the next token for each row, and write it into the token tensor, until the tensor is full.

There are a few conventions worth highlighting about how this function works:
1. We're going to use the convention that we generate up to a _total_ of `max_tokens` (including the prompt), so that the returned tensor is always `num_completions x max_tokens` in size.
2. For the `eot_token`, you should end generation only if _all_ the different completions contain the `eot_token`.  You can keep generating tokens as usual after the `eot_token` in the rows that have already finished, while you wait for the rest to be generated.

Note that you can use the `torch.multinomial(probs, 1)` function to generate from _multiple_ different distributions.  That is, if you have something like this:
```
probs = torch.tensor([
    [0.1, 0.2, 0.7],
    [0.6, 0.3, 0.1]
])
torch.multinomial(probs, 1)
```
then this will generate two samples, one from the probability distribution in the first row, and the second from the probability distribution in the second row.

In [ ]:
@mugrade.local_tests
def generate_parallel(model, prompt_tokens, num_completions = 1,
                      eot_token=None, temp=0.7, max_tokens=500):
    """
    Autoregressively sample multiple completions from a language model using its KV cache.

    Inputs:
        model : Module - language model mapping token sequences to logits
        prompt_tokens : list[int] - initial prompt tokens
        num_completions : int - number of different completions to generate
        eot_token: int or None - token at which generation may stop once all completions contain it
        temp : float - sampling temperature
        max_tokens : int - maximum total number of tokens per completion, including the prompt
    Output:
        torch.Tensor[int] (num_completions x max_tokens) - prompt plus generated tokens for each completion
    """
    ### BEGIN YOUR CODE
    device = next(model.parameters()).device
    prompt_len = len(prompt_tokens)
    tokens = torch.zeros(num_completions, max_tokens, dtype=torch.long, device=device) # (num_completions x max_tokens)
    tokens[:, :prompt_len] = torch.tensor(prompt_tokens, dtype=torch.long, device=device)
    input_tokens = tokens[:, :prompt_len] # (num_completions x prompt_len)
    seq_pos = 0
    with torch.no_grad():
        for pos in range(prompt_len, max_tokens):
            logits = model(input_tokens, seq_pos=seq_pos, use_kv_cache=True) # (num_completions x seq_len x num_tokens)
            seq_pos += input_tokens.shape[1]
            probs = torch.softmax(logits[:, -1].float() / temp, dim=-1) # (num_completions x num_tokens)
            tokens[:, pos] = torch.multinomial(probs, 1)[:, 0] # one sample per completion
            # only stop early once _every_ completion has generated the end token
            if eot_token is not None and (tokens[:, prompt_len:pos+1] == eot_token).any(dim=1).all():
                break
            input_tokens = tokens[:, pos:pos+1] # (num_completions x 1)
    return tokens
    ### END YOUR CODE

Let's now evaluate how well this works.  First run the following cell to download the necessary datasets and models for the rest of this assignment.

In [ ]:
### Download the models and datasets for the assignment
from huggingface_hub import hf_hub_download

repo = "zkolter/RL-Homework"
filenames = ["model_base.pth",
             "model_sft.pth",
             "params.json"]

for filename in filenames:
    if not os.path.exists(filename):
        hf_hub_download(repo_id=repo, filename=filename, repo_type="model", local_dir=".")


## download GSM8K
if not os.path.exists("gsm8k_train.json") or not os.path.exists("gsm8k_test.json"):
    data = load_dataset("openai/gsm8k", "main")
    # data = load_dataset("gsm8k", "main")
    with open("gsm8k_train.json", "wt") as f:
        json.dump(list(data["train"]), f, indent=4)
    with open("gsm8k_test.json", "wt") as f:
        json.dump(list(data["test"]), f, indent=4)

The following code will load the base model that we pretrained for you.  It uses exactly the architecture you built above, and was trained on roughly 1.1 trillion tokens of the [FineWebEDU](https://huggingface.co/datasets/HuggingFaceFW/fineweb-edu) dataset; the cell after it then generates several completions of a prompt in parallel.

**Note:** for the majority of this assignment, we're going to directly use CUDA for all actual model training, etc.  You can still evaluate test cases on a CPU implementation (to get full credit), but there's not much point to creating the fine-tuned version with only a few steps of optimization, as they will not work for the following portions.  When needed, we will provide variants of the models trained to that point where you can pass all the test cases, but we strongly recommend getting the GPU instance of Colab for this assignment (all full training runs will take less than 10 minutes).

In [ ]:
### load the model and test generation
with open("params.json", "rt") as f:
    params = json.load(f)
model = LLM(
    params["num_tokens"],
    params["dim"],
    params["n_heads"],
    params["max_seq_len"],
    params["ffn_dim"],
    params["n_layers"]
)
model.load_weights("model_base.pth")
model.float()
# model.cuda()

In [ ]:
### generate some text
base_tokenizer = tiktoken.get_encoding("gpt2")
prompt = "In a shocking discovery,"
tokens = generate_parallel(model, base_tokenizer.encode(prompt,allowed_special="all"), num_completions=4, temp=0.4, max_tokens=100)
for t in tokens:
    print(base_tokenizer.decode(t.tolist()), "\n")

## Part 2 - Building the GSM8K reasoning dataset

In this part of the assignment, you will convert the GSM8K dataset into the tagged format that both the finetuned model we provide and your own RL training in Part 4 are built around.  The [GSM8K](https://huggingface.co/datasets/openai/gsm8k) dataset contains a collection of simple math problems that require a few steps of reasoning in order to arrive at the right answer.  The dataset's entries consist of items like this:
```json
{
    "question": "Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?",
    "answer": "Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72"
}
```
There are a few points to highlight here.  First, as indicated, the objects above have "question" and "answer" as the main keys of the json dictionary, but there is also substantial structure within the "answer" portion in particular.  Specifically, the answer portions are broken down into the following elements:
1. Within the answer text, there are tool calls indicated by the `<< >>` tags, e.g., `<<48/2=24>>`.  In this format, the left hand side of the equation (less of the = sign) indicates the arithmetic expression to send the tool, and the element to the right represents the result of the tool call.  For instance, if you parsed this text you would input `48/2` into the tool (i.e., into a Python interpreter) and the expected result would be `24` (note that when you actually generate these results, your interpreter might generate 24.0 as the answer, or have other floating point issues, but you don't need to worry about this for our purposes).
2. The answer to the question appears always as an integer and always after the "`#### `" symbol on its own line.

### Converting GSM8K to our format

As a first task, you'll convert the GSM8K format to a more explicit format that we can parse with our tokenizer for use with SFT and RL training.  Specifically we will use three different tags: `<QUESTION>` to specify the question, `<THINK>` to detail the solution logic, and `<ANSWER>` to indicate the integer answer alone.  In addition within the `<THINK>` tags you can use the `<TOOL>` and `<RESPONSE>` tags to indicate the call to and return from a tool call.  For example, the above format would be parsed into
```
<QUESTION>Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?</QUESTION><THINK>Natalia sold 48/2 = <TOOL>48/2</TOOL><RESPONSE>24</RESPONSE>24 clips in May.\nNatalia sold 48+24 = <TOOL>48+24</TOOL><RESPONSE>72</RESPONSE>72 clips altogether in April and May.</THINK><ANSWER>72</ANSWER>
```

In [ ]:
@mugrade.local_tests
def gsm8k_to_text(message):
    """
    Convert one GSM8K example into the tagged reasoning format used in this homework.

    Input:
        message : dict[str, str] - GSM8K example with "question" and "answer" fields
    Output:
        str - formatted text containing QUESTION, THINK, TOOL, RESPONSE, and ANSWER tags
    """
    ### BEGIN YOUR CODE
    think, answer = message["answer"].rsplit("####", 1)
    # each <<expression=result>> tool call becomes an explicit tool call and response.
    # re.sub() is used to find all occurrences of the pattern <<expression=result>> in the think string 
    # and replace them with <TOOL>expression</TOOL><RESPONSE>result</RESPONSE>
    think = re.sub(r"<<(.*?)=(.*?)>>", r"<TOOL>\1</TOOL><RESPONSE>\2</RESPONSE>", think)
    return (f"<QUESTION>{message['question']}</QUESTION>"
            f"<THINK>{think.strip()}</THINK>" #.strip() to remove leading/trailing whitespace
            f"<ANSWER>{answer.strip()}</ANSWER>")
    ### END YOUR CODE

### Pretokenizing dataset

Rather than convert and tokenize the raw data during training, it's much more convenient to create a pre-tokenized version of the dataset once, and then have the data loader operate on that tokenized representation directly.

Implement the `pretokenize_gsm8k()` function, which reads one of the GSM8K json files (a list of examples in the original `{"question": ..., "answer": ...}` format), converts each example to the tagged text format using the `gsm8k_to_text()` function you just wrote, tokenizes it, and writes out another json file containing a list of the tokenized examples (i.e., a list of lists of integer tokens).

You'll want to tokenize the text using the call
```python
tokenizer.encode(text, allowed_special="all")
```
where the `allowed_special` argument ensures that our special tags are tokenized to their own special ids, instead of being broken up into ordinary text tokens.  Use `json.load()` and `json.dump()` (with the optional argument `indent=4` if you want prettier outputs) to load and save the files.

In [ ]:
@mugrade.local_tests
def pretokenize_gsm8k(tokenizer, in_filename, out_filename):
    """
    Convert GSM8K json examples into tokenized tagged reasoning traces.

    Inputs:
        tokenizer : object - tokenizer with encode(text, allowed_special="all")
        in_filename : str - input json filename containing GSM8K examples
        out_filename : str - output json filename for tokenized examples
    Output:
        None - writes the tokenized examples to out_filename
    """
    ### BEGIN YOUR CODE
    with open(in_filename, "rt") as f:
        messages = json.load(f)
    tokens = [tokenizer.encode(gsm8k_to_text(message), allowed_special="all") for message in messages]
    with open(out_filename, "wt") as f:
        json.dump(tokens, f, indent=4)
    ### END YOUR CODE

### Loss masking

Implement the `get_loss_mask()` function, which returns a binary mask of which tokens to train on for our GSM8K format.  The convention here is that the mask should start off false (you don't train on the question portion up to the `<THINK>` token), be true after the `<THINK>` token (this is where the LLM starts generating the answer), be false again after any `</TOOL>` token (you don't try to predict tool response), and then be true again after any `</RESPONSE>` token (after the tool call you once again start generating text).  Finally, anything after `</ANSWER>` (if it exists) would also be false.

For example for the text above,
```
<QUESTION>Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?</QUESTION><THINK>Natalia sold 48/2 = <TOOL>48/2</TOOL><RESPONSE>24</RESPONSE>24 clips in May.\nNatalia sold 48+24 = <TOOL>48+24</TOOL><RESPONSE>72</RESPONSE>72 clips altogether in April and May.</THINK><ANSWER>72</ANSWER>
```
the mask would be true corresponding to the following tokens:
```
Natalia sold 48/2 = <TOOL>48/2</TOOL>24 clips in May.\nNatalia sold 48+24 = <TOOL>48+24</TOOL>72 clips altogether in April and May.</THINK><ANSWER>72</ANSWER>
```

In [ ]:
@mugrade.local_tests
def get_loss_mask(tokens, tokenizer):
    """
    Build a boolean mask selecting reasoning and answer tokens to train on for GSM8K.

    Inputs:
        tokens : list[int] - tokenized GSM8K example with QUESTION, THINK, TOOL, RESPONSE, and ANSWER tags
        tokenizer : object - tokenizer with special token ids for the GSM8K tags
    Output:
        list[bool] - True on generated reasoning and answer tokens, but False on questions and tool responses
    """
    ### BEGIN YOUR CODE
    mask = []
    train_on_token = False
    for token in tokens:
        # the tag tokens themselves keep the state they were generated under, and only
        # change the state for the tokens that follow them
        mask.append(train_on_token)
        if token == tokenizer._special_tokens["<THINK>"]:
            train_on_token = True # the model generates everything after the question
        elif token == tokenizer._special_tokens["</TOOL>"]:
            train_on_token = False # the tool, not the model, generates the response
        elif token == tokenizer._special_tokens["</RESPONSE>"]:
            train_on_token = True # back to the model after the tool response
        elif token == tokenizer._special_tokens["</ANSWER>"]:
            train_on_token = False # nothing to generate after the answer
    return mask
    ### END YOUR CODE

If you have implemented these functions correctly, then the following code will generate the tokenized files you use in the remainder of this homework.  Note that this also builds the tokenizer we use throughout the assignment, explicitly adding our special tags as special tokens, so that these elements always tokenize to their own single ids.

In [ ]:
base_tokenizer = tiktoken.get_encoding("gpt2")
tokenizer = tiktoken.Encoding(
    name="gpt2_chat",
    pat_str=base_tokenizer._pat_str,
    mergeable_ranks=base_tokenizer._mergeable_ranks,
    special_tokens={**base_tokenizer._special_tokens,
                    "<QUESTION>": 50257,
                    "</QUESTION>": 50258,
                    "<THINK>": 50259,
                    "</THINK>": 50260,
                    "<TOOL>": 50261,
                    "</TOOL>": 50262,
                    "<RESPONSE>": 50263,
                    "</RESPONSE>": 50264,
                    "<ANSWER>": 50265,
                    "</ANSWER>": 50266}
)

pretokenize_gsm8k(tokenizer, "gsm8k_train.json", "gsm8k_train_tokenized.json")
pretokenize_gsm8k(tokenizer, "gsm8k_test.json", "gsm8k_test_tokenized.json")

### Data loader for GSM8K

Next, write a data loader that reads the (pretokenized) GSM8K data and returns it in a form suitable for training.  This loader simply loads the entire pretokenized json file into memory and forms each batch on the fly.  As with any [Python iterator](https://www.w3schools.com/python/python_iterators.asp), in addition to the `__init__()` function you have to implement an `__iter__()` function (which resets the iteration and returns `self`) and a `__next__()` function (which returns the next batch, or raises `StopIteration` once the data is exhausted).

Each batch consists of _three_ tensors
```python
for x, y, mask in loader:
    ...
```
where `x` contains the input tokens, `y` contains the same tokens shifted by one (i.e., the next-token targets), and `mask` is a boolean tensor of the same size as `y`, formed by the `get_loss_mask()` function above, which indicates which target tokens we actually want to train on.  Each element of a batch consists of a single GSM8K example, zero-padded at the end, with `False` entries in the mask over the padding so that these tokens are ignored by the loss.  You should also drop incomplete batches, i.e., batches which have size < `batch_size`.

One thing to be careful about: rather than specify a maximum sequence length and always return objects of that size, since GSM8K data is often much shorter, you should instead dynamically compute the maximum sequence length over the examples in the _current_ batch, and only return `x`, `y`, and `mask` up to that sequence length (really one minus that maximum sequence length, since `x` will be all tokens up to the max length minus one, and `y` and `mask` start at the first predicted token).

In [ ]:
@mugrade.local_tests
class DataLoader:
    def __init__(self, filename, batch_size, tokenizer, device="cpu"):
        """
        Initialize a GSM8K data loader backed by a tokenized json file.

        Inputs:
            filename : str - json filename containing tokenized GSM8K examples
            batch_size : int - number of examples per minibatch
            tokenizer : object - tokenizer used to build reasoning and answer masks
            device : str - device on which to place each minibatch tensor
        """
        ### BEGIN YOUR CODE
        with open(filename, "rt") as f:
            self.examples = json.load(f)
        self.batch_size = batch_size
        self.tokenizer = tokenizer
        self.device = device
        self.num_batches = len(self.examples) // batch_size # drop incomplete batches
        self.current_batch = 0
        ### END YOUR CODE

    def __iter__(self):
        """
        Reset iteration state and return the iterator object.

        Output:
            DataLoader - iterator over GSM8K minibatches
        """
        ### BEGIN YOUR CODE
        self.current_batch = 0
        return self
        ### END YOUR CODE

    def __next__(self):
        """
        Return the next GSM8K minibatch padded only to this batch's maximum length.

        Output:
            tuple(torch.Tensor, torch.Tensor, torch.Tensor) - input tokens, next-token targets, and boolean loss mask
        """
        ### BEGIN YOUR CODE
        if self.current_batch >= self.num_batches:
            raise StopIteration
        start = self.current_batch * self.batch_size
        self.current_batch += 1
        examples = self.examples[start:start + self.batch_size]

        # pad only out to the longest example in this batch, rather than to a fixed sequence length
        max_len = max(len(example) for example in examples)
        tokens = torch.zeros(self.batch_size, max_len, dtype=torch.long)
        mask = torch.zeros(self.batch_size, max_len, dtype=torch.bool)
        for i, example in enumerate(examples):
            tokens[i, :len(example)] = torch.tensor(example, dtype=torch.long)
            mask[i, :len(example)] = torch.tensor(get_loss_mask(example, self.tokenizer), dtype=torch.bool)
        tokens = tokens.to(self.device)
        mask = mask.to(self.device)
        return tokens[:, :-1], tokens[:, 1:], mask[:, 1:]
        ### END YOUR CODE

### The SFT starting point

RL only works if the model already produces roughly the right format often enough that some completions earn reward, so rather than starting from the base model we start from `model_sft.pth`.  We produced this checkpoint by supervised finetuning the base model for 5 epochs on exactly the tagged data you just built, using a next-token loss restricted to the reasoning and answer tokens that your `get_loss_mask()` selects.  It was downloaded along with the base model above, and both the evaluation below and the RL training in Part 4 load it directly.


## Part 3 - Evaluating tools and reasoning models

Evaluating a reasoning model with tool calls (especially on a concrete task like GSM8K) is not as simple as just generating text from the model and reading off the result.  Instead, we need to generate text, intercept tool call tags as needed, run the actual Python interpreter to generate the output of the tools, insert this as a tool response, and when finished, validate the final answer versus the ground truth answer in a test set.  And most of this same logic will be needed to train the RL methods themselves.  This part of the homework will build up these elements bit by bit.

### Tool evaluation

As a starting point, you'll need to design a tool evaluation function that can evaluate simple arithmetic expressions like `48/2` and `24+28` that are included in the example above.  For this assignment we'll take a (quite risky) approach and just directly evaluate these expressions using the Python `eval()` function.  However, because GSM8K uses integers as the results of tool calls when the answer is an integer and _also_ has some intermediate calculations that produce decimal results (versus Python, which will always return floating point solutions when e.g., dividing two integers), you also need to convert the final answer to an integer _if it is close to an integer_.  Specifically, your function should work as follows:
1. Call `eval()` on `tool_call_text` to parse the result.  By the conventions of GSM8K, if the tool call is properly formatted, the result will always be a number (either an integer or a floating point number).
2. Check if the result is within `1e-4` of an integer, which you can compute using the `math.isclose(..., abs_tol=1e-4)` function along with the built-in Python `round()` function.  If so, then return this integer value instead of the floating point value.
3. There is, of course, the chance that your LLM will not generate valid code within the tool call.  To account for this, wrap _all_ of this logic within a `try`/`except` block, and set the response to be an error if there is any exception.  In other words, this works like the following:
```python
try:
    ### do the evaluation
except:
    response = "ERROR"
```

In [ ]:
@mugrade.local_tests
def eval_tool(tool_call_text):
    """
    Evaluate an arithmetic tool call, rounding near-integer results and catching failures.

    Input:
        tool_call_text : str - arithmetic expression generated between TOOL tags
    Output:
        int, float, or str - evaluated result, rounded integer when appropriate, or "ERROR" on failure
    """
    ### BEGIN YOUR CODE
    try:
        response = eval(tool_call_text)
        # GSM8K reports integer results, but Python division always returns a float
        if math.isclose(response, round(response), abs_tol=1e-4):
            response = round(response)
    except:
        response = "ERROR"
    return response
    ### END YOUR CODE

### Generation with tool calls

Next, modify the `generate_parallel()` function you wrote above to include the results of tool calls.  This is a somewhat involved process, so we break down each element that you will need to do:
1. Copy the prompt to the needed number of completions and start generating text as in the original `generate_parallel()` call.
2. If you encounter a `</TOOL>` token in any of the batches, then extract the tool call text by searching for the most recent `<TOOL>` token, using the tokenizer to decode the text between these tags, and evaluate the tool call (using the `eval_tool()` function).  If no opening `<TOOL>` tag exists then you should generate an `ERROR` response.
3. Include the tool response inside `<RESPONSE>` `</RESPONSE>` tags, and tokenize the text into a sequence of tokens (let's say that the total length of these tokens is `n`).  Now, when generating the next `n` tokens for this particular element in the batch, don't include the sample produced by `torch.multinomial()`, but directly overwrite it with the tokenized response (and after you have written these `n` tokens, go back to generating text as normal).
4. If you ever encounter an `</ANSWER>` tag in one of the batch elements, force all tokens to be zero for this particular element after that point.

Note that all these tool calls need to be done individually for each generated completion (not all at once over all batch elements), because each generation will likely make tool calls at different points in the generation, etc.

In [ ]:
@mugrade.local_tests
def generate_parallel_tool(model, prompt_tokens, tokenizer, num_completions = 1,
                           eot_token=None, temp=0.7, max_tokens=500):
    """
    Autoregressively sample multiple completions and inject tool responses when needed.

    Inputs:
        model : Module - language model mapping token sequences to logits
        prompt_tokens : list[int] - initial prompt tokens
        tokenizer : object - tokenizer with encode() and decode() methods
        num_completions : int - number of different completions to generate
        eot_token: int or None - token at which generation may stop once all completions contain it
        temp : float - sampling temperature
        max_tokens : int - maximum total number of tokens in each prompt and completion
    Output:
        torch.Tensor[int] (num_completions x max_tokens) - prompt plus generated tokens for each completion
    """
    ### BEGIN YOUR CODE
    device = next(model.parameters()).device
    prompt_len = len(prompt_tokens)
    tokens = torch.zeros(num_completions, max_tokens, dtype=torch.long, device=device) # (num_completions x max_tokens)
    tokens[:, :prompt_len] = torch.tensor(prompt_tokens, dtype=torch.long, device=device)
    input_tokens = tokens[:, :prompt_len] # (num_completions x prompt_len)
    seq_pos = 0

    tool_open = tokenizer._special_tokens["<TOOL>"]
    tool_close = tokenizer._special_tokens["</TOOL>"]
    answer_close = tokenizer._special_tokens["</ANSWER>"]
    responses = [[] for _ in range(num_completions)] # tool response tokens still to be written
    done = [False] * num_completions # completions that have already generated their answer

    with torch.no_grad():
        for pos in range(prompt_len, max_tokens):
            logits = model(input_tokens, seq_pos=seq_pos, use_kv_cache=True) # (num_completions x seq_len x num_tokens)
            seq_pos += input_tokens.shape[1]
            probs = torch.softmax(logits[:, -1].float() / temp, dim=-1) # (num_completions x num_tokens)
            next_tokens = torch.multinomial(probs, 1)[:, 0] # (num_completions)

            # tool responses (and the padding after an answer) overwrite the sampled token
            for i in range(num_completions):
                if done[i]:
                    next_tokens[i] = 0
                elif responses[i]:
                    next_tokens[i] = responses[i].pop(0) # pop out the next token from the tool response
            tokens[:, pos] = next_tokens

            for i in range(num_completions):
                token = tokens[i, pos].item()
                if token == answer_close:
                    done[i] = True
                elif token == tool_close:
                    # run the tool on the text back to the most recent <TOOL> tag
                    row = tokens[i, :pos].tolist()
                    if tool_open in row:
                        start = len(row) - 1 - row[::-1].index(tool_open)
                        result = eval_tool(tokenizer.decode(row[start+1:]))
                    else:
                        result = "ERROR"
                    responses[i] = tokenizer.encode(f"<RESPONSE>{result}</RESPONSE>", allowed_special="all")

            # only stop early once _every_ completion has generated the end token
            if eot_token is not None and (tokens[:, prompt_len:pos+1] == eot_token).any(dim=1).all():
                break
            input_tokens = tokens[:, pos:pos+1] # (num_completions x 1)
    return tokens
    ### END YOUR CODE

### Extracting and grading answers

Implement the following two functions below:
1. `extract_answer()` should search for any text contained between the `<ANSWER>` and `</ANSWER>` tags and convert this to an _integer_ value.  If any part of this process fails (e.g., if there are no answer tags, or if the content between them does not parse to a valid integer), then the function should return `None`.
2. `grade_responses()` should take a 2D tensor of tokens representing multiple different generations, extract the answer of each, and compare it to the ground truth answer.  The result should be a list of response grades, where each grade is given by `correct_weight` (if the response is correct) added to `format_weight` (if the response is properly formatted, i.e., `extract_answer()` does not return `None`).  These parameters will help us evaluate proper formatting versus correct answers for the generated responses, and also be useful in reinforcement learning for designing reward functions that take these different elements into account.

In [ ]:
@mugrade.local_tests
def extract_answer(tokenizer, tokens):
    """
    Extract the integer answer contained between ANSWER tags.

    Inputs:
        tokenizer : object - tokenizer with decode() and special token ids for the ANSWER tags
        tokens : list[int] - token sequence potentially containing an ANSWER span
    Output:
        int or None - parsed integer answer, or None if extraction fails
    """
    ### BEGIN YOUR CODE
    try:
        tokens = [int(token) for token in tokens]
        start = tokens.index(tokenizer._special_tokens["<ANSWER>"])
        end = tokens.index(tokenizer._special_tokens["</ANSWER>"], start)
        return int(tokenizer.decode(tokens[start+1:end]))
    except:
        return None
    ### END YOUR CODE

In [ ]:
@mugrade.local_tests
def grade_responses(tokenizer, tokens, ground_truth,
                    correct_weight=1.0, format_weight = 0.0):
    """
    Score multiple generated responses by correctness and answer formatting.

    Inputs:
        tokenizer : object - tokenizer used to decode answers from token sequences
        tokens : torch.Tensor[int] (num_completions x seq_len) - generated completions to score
        ground_truth : torch.Tensor[int] (seq_len,) - reference token sequence containing the correct answer
        correct_weight : float - reward added when a completion's answer matches the ground truth
        format_weight : float - reward added when a completion contains a valid integer ANSWER span
    Output:
        list[float] - one score per completion
    """
    ### BEGIN YOUR CODE
    correct_answer = extract_answer(tokenizer, ground_truth)
    grades = []
    for row in tokens:
        answer = extract_answer(tokenizer, row)
        grade = 0.0
        if answer is not None:
            grade += format_weight
            if answer == correct_answer:
                grade += correct_weight
        grades.append(grade)
    return grades
    ### END YOUR CODE

### Evaluating the reasoning model

Finally, put all of these elements together to write a function that evaluates the accuracy of a reasoning model on GSM8K tasks.  Your function should take a model and a data loader (with batch size 1), and for each item in the loader, you should extract the text up to and including the `<THINK>` token and treat this as the prompt to the model.  Then, generate `num_completions` different completions to the prompt and for each one, grade the number of correct and properly formatted answers.

Your function should return three elements:
1. The fraction of correct answers out of all `max_cases * num_completions` that you generate (this approximates the true probability of the model's accuracy, or the Pass@1 rate).
2. The fraction of all answers, again out of all `max_cases * num_completions`, that are properly formatted.
3. The fraction of all examples that had at least _one_ correct answer.  This represents what's referred to as the Pass@k rate, where here `k` is the number of completions.

In [ ]:
@mugrade.local_tests
def eval_model(loader, model, tokenizer,
               num_completions=32, max_tokens=200, temp=0.6, max_cases=100):
    """
    Evaluate a reasoning model by sampling multiple completions for each GSM8K prompt.

    Inputs:
        loader : iterable - GSM8K data loader with batch_size equal to 1
        model : Module - language model to evaluate
        tokenizer : object - tokenizer with the GSM8K special token ids
        num_completions : int - number of completions to generate per prompt
        max_tokens : int - maximum total number of tokens per completion
        temp : float - sampling temperature
        max_cases : int - maximum number of examples from the loader to evaluate
    Output:
        tuple(float, float, float) - pass@1 accuracy, formatting rate, and pass@k rate
    """
    ### BEGIN YOUR CODE
    num_cases, num_correct, num_formatted, num_passk = 0, 0.0, 0.0, 0
    for i, (x, y, mask) in enumerate(loader):
        if i >= max_cases:
            break
        # prompt the model with everything up to and including the <THINK> token
        prompt = x[0].tolist()
        prompt = prompt[:prompt.index(tokenizer._special_tokens["<THINK>"]) + 1]
        tokens = generate_parallel_tool(model, prompt, tokenizer,
                                        num_completions=num_completions,
                                        eot_token=tokenizer._special_tokens["</ANSWER>"],
                                        temp=temp, max_tokens=max_tokens) # (num_completions x max_tokens)
        correct = grade_responses(tokenizer, tokens, y[0], correct_weight=1.0, format_weight=0.0)
        formatted = grade_responses(tokenizer, tokens, y[0], correct_weight=0.0, format_weight=1.0)
        num_correct += sum(correct)
        num_formatted += sum(formatted)
        num_passk += sum(correct) > 0 # at least one correct completion for this question
        num_cases += 1
    return (num_correct / (num_cases * num_completions),
            num_formatted / (num_cases * num_completions),
            num_passk / num_cases)
    ### END YOUR CODE

After implementing all of these functions, you can now evaluate the supervised-finetuned model, which is the baseline that RL training in Part 4 has to improve upon.  Given the fact that answers are generated randomly, there is going to be some variance in the numbers you'll see in these evaluations, but to give you a rough range of estimates, in our tests we have an accuracy of around 2-2.5%, a Pass@32 of 25-35% and a correct formatting of 70-80%.

In [ ]:
model.load_weights("model_sft.pth")
model.float().to(device)
test_loader = DataLoader("gsm8k_test_tokenized.json", 1, tokenizer, device=device)
acc, formatting, passk = eval_model(test_loader, model, tokenizer,
                                    num_completions=32, max_tokens=200, max_cases=100, temp=0.7)
print(f"Accuracy (Pass@1): {acc}\nPass@32: {passk}\nCorrect format: {formatting}")

## Part 4 - Reinforcement learning

Now that all the other components of a reasoning model are in place, let's build an RL training procedure.  The good news here is that the most difficult parts of writing an RL method (sampling, appropriate tool use, and grading) are already done, and so the actual RL training component can be extremely short.

### RL Loss

First implement the loss function for an RL training loop.  Given a prompt $x$ and several different completions $y_1,\ldots,y_N$, we define the RL loss as
$$
\mathcal{L}_{RL} = -\frac{1}{N_{tok}} \sum_{i=1}^N \log p(y_i | x) \cdot (R(x,y_i) - \bar{R})
$$
where $\log p(y_i|x)$ is the sum of log probabilities (on valid predicted tokens, i.e., tokens where the mask is `True`, as implemented by the `log_probs()` function), where $N_{tok}$ is the _total_ number of valid predicted tokens (i.e., the sum of `True` elements in the masks), and where $\bar{R}$ is the mean reward for this batch
$$
\bar{R} = \frac{1}{N} \sum_{j=1}^N R(x,y_j).
$$

This is the classical policy gradient (REINFORCE) objective with a mean-reward baseline: the gradient of this loss has the same expected value as the gradient of the true loss we care about, the (negative) expected reward under the completions generated by the model, while subtracting the mean reward $\bar{R}$ leaves that expectation unchanged but substantially reduces the variance of the estimate.  Intuitively, the loss increases the probability of completions that did better than average for this prompt, and decreases the probability of those that did worse.

Note that the mask here is computed with the same `get_loss_mask()` function you wrote for supervised finetuning, applied to the _generated_ token sequences: this is what keeps the model from being trained on the prompt, or on the tool responses that the tool (rather than the model) produced.

In [ ]:
@mugrade.local_tests
def rl_loss(model, tokenizer, tokens, rewards):
    """
    Compute the centered policy-gradient loss for multiple sampled completions.

    Inputs:
        model : Module - language model assigning logits to token sequences
        tokenizer : object - tokenizer used to build GSM8K loss masks
        tokens : torch.Tensor[int] (num_completions x seq_len) - prompt plus sampled completion tokens
        rewards : torch.Tensor[float] (num_completions,) - reward assigned to each sampled completion
    Output:
        torch.Tensor[float] () - scalar RL loss
    """
    ### BEGIN YOUR CODE
    mask = torch.tensor([get_loss_mask(row.tolist(), tokenizer) for row in tokens],
                        dtype=torch.bool, device=tokens.device) # (num_completions x seq_len)
    logits = model(tokens[:, :-1]) # (num_completions x seq_len-1 x num_tokens)
    logp = log_probs(logits.float(), tokens[:, 1:], mask[:, 1:]) # (num_completions)
    advantage = rewards - rewards.mean() # (num_completions)
    return -(logp * advantage).sum() / mask.sum()
    ### END YOUR CODE

### RL training

Finally, put this all together to build your RL training loop.  Like the evaluation function, `train_llm_rl()` takes a model and a data loader (which must have batch size 1), and then iterates over the whole loader or for a maximum of `max_iter` iterations.  For each example in the loader, it will actually _ignore_ the `y` and `mask` outputs of the loader, and instead just extract the prompt up to and including the `<THINK>` token, and generate `num_completions` completions with tool calls.  You should grade these completions with the `grade_responses()` function using the given `correct_weight` and `format_weight` parameters (this defines the reward so as to prioritize both correct answers and proper formatting), though you're welcome to experiment with these weights if you want.  Given these rewards, compute the RL loss and take an optimization step in the normal fashion.

In [ ]:
@mugrade.local_tests
def train_llm_rl(model, loader, opt, tokenizer,
                 num_completions=32, temp=0.8, max_tokens=200, max_iter=None,
                 correct_weight = 1.0, format_weight = 0.05):
    """
    Run one pass of reinforcement learning using sampled completions and scalar rewards.

    Inputs:
        model : Module - language model being optimized
        loader : iterable - GSM8K data loader with batch_size equal to 1
        opt : optimizer - object with zero_grad() and step() methods
        tokenizer : object - tokenizer with the GSM8K special token ids
        num_completions : int - number of completions to sample per prompt
        temp : float - sampling temperature
        max_tokens : int - maximum total number of tokens per sampled completion
        max_iter : int or None - maximum number of loader iterations to process
        correct_weight : float - reward added when a completion is correct
        format_weight : float - reward added when a completion has a valid ANSWER span
    Output:
        None - updates model parameters in place
    """
    ### BEGIN YOUR CODE
    for i, (x, y, mask) in enumerate(loader):
        if max_iter is not None and i >= max_iter:
            break
        # sample completions for everything up to and including the <THINK> token
        prompt = x[0].tolist()
        prompt = prompt[:prompt.index(tokenizer._special_tokens["<THINK>"]) + 1]
        tokens = generate_parallel_tool(model, prompt, tokenizer,
                                        num_completions=num_completions,
                                        eot_token=tokenizer._special_tokens["</ANSWER>"],
                                        temp=temp, max_tokens=max_tokens) # (num_completions x max_tokens)
        # the reward rewards both correct answers and properly formatted ones
        rewards = torch.tensor(grade_responses(tokenizer, tokens, y[0],
                                               correct_weight=correct_weight,
                                               format_weight=format_weight),
                               device=tokens.device) # (num_completions)
        loss = rl_loss(model, tokenizer, tokens, rewards)
        opt.zero_grad()
        loss.backward()
        opt.step()
        print(f"iteration {i}: mean reward {rewards.mean().item():.4f}, loss {loss.item():.4f}")
    ### END YOUR CODE

If you have implemented all of this correctly, you can use the following code to further train the SFT'd model using RL.  As should be apparent, RL training is _substantially_ slower than supervised finetuning, and so we're only going to train on 200 total iterations here (the additional performance you'll get from this weaker model will plateau pretty quickly anyway).  This process should take between 2-5 minutes on GPUs.

In [ ]:
train_loader = DataLoader("gsm8k_train_tokenized.json", 1, tokenizer, device=device)
model.load_weights("model_sft.pth")
model.float().to(device)
opt = Adam(model.parameters(), lr=2e-6, betas=(0.9, 0.95))
train_llm_rl(model, train_loader, opt, tokenizer, num_completions=32, max_iter=200)

And finally, we can evaluate the performance of the resulting model.  Here there is going to be a _huge_ amount of potential variation, including the possibility that some runs may end up with worse performance than the SFT model, just due to bad luck from the sampling process over the course of RL.  However, in general we find that our method here gets 3-3.5% accuracy, 35-40% Pass@32, and 90+% formatting accuracy, which is a substantial relative increase over the SFT model alone (and remember, this is all evaluated on a test set different from those that the model was trained upon).

In [ ]:
test_loader = DataLoader("gsm8k_test_tokenized.json", 1, tokenizer, device=device)
acc, formatting, passk = eval_model(test_loader, model, tokenizer,
                                    num_completions=32, max_tokens=200, max_cases=100, temp=0.7)
print(f"Accuracy (Pass@1): {acc}\nPass@32: {passk}\nCorrect format: {formatting}")